In [4]:
# ============================================================
# STEP 1: Load and Explore Dataset
# ============================================================
print("\nSTEP 1: Loading and Exploring the dataset.....")
import pandas as pd
import numpy as np

# Load Fake and Real news datasets
fake = pd.read_csv("Fake.csv")
real = pd.read_csv("True.csv")

print('\n The Fake and Real csv files are imported')

# Display first few rows
print("\nFirst 3 rows of Fake News dataset: ")
display(fake.head(3))

print("\n First 3 rows of Real News dataset:")
display(real.head(3))

# Add Lables: 1 = Fake, 0 = Real 
fake["label"] = 1
real["label"] = 0

# Combine Datasets
df = pd.concat([fake, real])[["text", "label"]]

# Show dataset distribution
print("\nNumber of Fake vs Real samples:")
print(df["label"].value_counts())
print("\n0 = Real News, 1 = Fake News")


STEP 1: Loading and Exploring the dataset.....

 The Fake and Real csv files are imported

First 3 rows of Fake News dataset: 


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"



 First 3 rows of Real News dataset:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"



Number of Fake vs Real samples:
label
1    23481
0    21417
Name: count, dtype: int64

0 = Real News, 1 = Fake News


In [5]:
# ============================================================
# STEP 2: Text Preprocessing
# ============================================================

print("\nSTEP 2: Converting text to numerical features using TF-IDF.....")

from sklearn.feature_extraction.text import TfidfVectorizer

# TF_IDF Vectorization
vectorizer = TfidfVectorizer(max_features=3000, stop_words="english")
X = vectorizer.fit_transform(df["text"]).toarray()
y = df["label"].values

print("Text Successfully converted into numerical features !")
print("Feature matrix shape:", X.shape)


STEP 2: Converting text to numerical features using TF-IDF.....
Text Successfully converted into numerical features !
Feature matrix shape: (44898, 3000)


In [8]:
# ============================================================
# STEP 3: Split Dataset
# ============================================================

print("\nSTEP 3: Splitting dataset into training and testing sets(80/20).....")

from sklearn.model_selection import train_test_split
import torch

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

# Convert to pyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

print( f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")


STEP 3: Splitting dataset into training and testing sets(80/20).....
Training samples: 35918, Testing samples: 8980


In [11]:
# ============================================================
# STEP 4: Build Feedforward Neural Network
# ============================================================

print("\nSTEP 4: Creating the Neural Network....")

import torch.nn as nn 

class FakeNewsNN(nn.Module):
    def __init__(self, input_size):
        super(FakeNewsNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64,1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return X
input_size = X_train.shape[1]
model = FakeNewsNN(input_size)

print("Neural network architecture")
print(model)


STEP 4: Creating the Neural Network....
Neural network architecture
FakeNewsNN(
  (fc1): Linear(in_features=3000, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [ ]:
# ============================================================
# STEP 5: Train the Model
# ============================================================

print("\nSTEP 5: Training the neural network....")

import torch.optim as optim
import matplotlib.pyplot as plt

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15
losses = []

for epoch in range(epochs):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    print(f"Epoch {epoch+1}/{epoches} - loss: {loss.item(): .4f}")

# plot training Loss
plt.plot(losses)
plt.title("Training Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()